[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C13_RL_Foundations_Course/01_mdp_bellman/01_mdp_bellman.ipynb)

# 01 · MDP 与 Bellman 方程（纯 numpy）

目标：把 **MDP**、**Bellman 期望/最优方程**、**价值迭代(VI)**、**策略迭代(PI)** 从零实现，在随机 **GridWorld** 上解出最优策略，并对拍精确解。

路线：GridWorld → (P,R) 张量 → 策略评估(解析+迭代) → 价值迭代 + γ^k 收敛曲线 → 策略迭代 → VI/PI 对拍 → ✏️ 练习 → 📖 答案 → 🧪 经典基准胶囊。

> 心智模型：**Bellman = 即时奖励 + 折扣后继价值**；**VI 反复取 max 的备份**，**PI 评估+改进交替**，两者收敛到**同一个** $V^*$。GridWorld 状态少到能对拍解析真值。

## 1 · 把随机 GridWorld 写成 (P, R) 张量

经典 Russell & Norvig 4×3 GridWorld：3 行 4 列，目标 `(0,3)=+1`，陷阱 `(1,3)=-1`，墙 `(1,1)`，每步 `-0.04`。

**随机性**：意图动作以 0.8 成功，各 0.1 滑向**垂直**方向；撞墙/边界停在原地。动作 `0/1/2/3 = 上/右/下/左`。

我们把它编码成转移张量 `P[s,a,s']`（形状 `(S,A,S)`，每行是概率分布）和期望奖励 `R[s,a]`（形状 `(S,A)`）。

In [ ]:
import numpy as np

class GridWorld:
    def __init__(self, gamma=0.9, slip=0.2, step_reward=-0.04):
        self.n_rows, self.n_cols = 3, 4
        self.obstacles = {(1, 1)}
        self.terminals = {(0, 3): 1.0, (1, 3): -1.0}
        self.step_reward = step_reward
        self.gamma = gamma
        self.slip = slip
        self.moves = {0: (-1, 0), 1: (0, 1), 2: (1, 0), 3: (0, -1)}  # 上右下左
        self.perp  = {0: (3, 1), 1: (0, 2), 2: (1, 3), 3: (2, 0)}    # 每动作的两个垂直方向
        self.states = [(r, c) for r in range(self.n_rows) for c in range(self.n_cols)
                       if (r, c) not in self.obstacles]
        self.S, self.A = len(self.states), 4
        self.sidx = {s: i for i, s in enumerate(self.states)}
    def is_terminal(self, s):
        return s in self.terminals
    def _move(self, s, a):
        dr, dc = self.moves[a]; r, c = s; nr, nc = r + dr, c + dc
        if (nr, nc) in self.obstacles or not (0 <= nr < self.n_rows and 0 <= nc < self.n_cols):
            return s              # 撞墙/出界：停在原地
        return (nr, nc)
    def transitions(self, s, a):
        '''返回 [(prob, s', reward), ...]。'''
        if self.is_terminal(s):
            return [(1.0, s, 0.0)]     # 终止态自环、奖励 0
        out = {}
        la, ra = self.perp[a]
        for prob, act in [(1 - self.slip, a), (self.slip / 2, la), (self.slip / 2, ra)]:
            sp = self._move(s, act)
            out[sp] = out.get(sp, 0.0) + prob
        res = []
        for sp, prob in out.items():
            r = self.terminals[sp] if sp in self.terminals else self.step_reward
            res.append((prob, sp, r))
        return res

def build_PR(env):
    S, A = env.S, env.A
    P = np.zeros((S, A, S)); R = np.zeros((S, A))
    for s in env.states:
        i = env.sidx[s]
        for a in range(A):
            for prob, sp, r in env.transitions(s, a):
                P[i, a, env.sidx[sp]] += prob
                R[i, a] += prob * r        # 期望奖励
    return P, R

env = GridWorld()
P, R = build_PR(env)
print(f'状态数 S={env.S}, 动作数 A={env.A}')
print('P 形状', P.shape, '| R 形状', R.shape)
# 每个 (s,a) 的转移概率必须和为 1
assert np.allclose(P.sum(axis=2), 1.0), '每个 (s,a) 转移概率应和为 1'
# 验证随机性：起点 (2,0) 向上，0.8 到 (1,0)，0.1 滑左(撞墙留原地)，0.1 滑右到 (2,1)
i = env.sidx[(2, 0)]
probs = {env.states[j]: P[i, 0, j] for j in range(env.S) if P[i, 0, j] > 0}
print('从 (2,0) 选「上」的转移分布:', {k: round(v, 2) for k, v in probs.items()})
assert abs(probs[(1, 0)] - 0.8) < 1e-9
print('✅ GridWorld 编码为 (P,R) 张量，随机滑动正确')

## 2 · 策略评估（解析解）：解线性 Bellman 期望方程

给定策略 $\pi$，价值满足 $V^\pi = R^\pi + \gamma P^\pi V^\pi$，这是**线性**方程组，可直接求逆：

$$V^\pi = (I - \gamma P^\pi)^{-1} R^\pi$$

其中 $P^\pi_{s,s'} = \sum_a \pi(a|s) P(s'|s,a)$，$R^\pi_s = \sum_a \pi(a|s) R(s,a)$。先评估一个「永远向上」的确定性策略。

In [ ]:
def policy_to_matrix(pi_probs, P, R):
    '''pi_probs[s,a] = π(a|s)。返回 P_pi (S,S) 与 R_pi (S,)。'''
    P_pi = np.einsum('sa,sap->sp', pi_probs, P)   # Σ_a π(a|s) P(s'|s,a)
    R_pi = np.einsum('sa,sa->s', pi_probs, R)      # Σ_a π(a|s) R(s,a)
    return P_pi, R_pi

def policy_eval_analytic(pi_probs, P, R, gamma, term_mask):
    S = P.shape[0]
    P_pi, R_pi = policy_to_matrix(pi_probs, P, R)
    V = np.linalg.solve(np.eye(S) - gamma * P_pi, R_pi)   # (I-γP)^{-1} R
    V[term_mask] = 0.0                                    # 终止态价值钉为 0
    return V

term_mask = np.array([env.is_terminal(s) for s in env.states])
# 确定性策略「永远向上(0)」编码成 one-hot 概率
pi_up = np.zeros((env.S, env.A)); pi_up[:, 0] = 1.0
V_up = policy_eval_analytic(pi_up, P, R, env.gamma, term_mask)
print('策略「永远向上」的价值 V^π：')
for r in range(env.n_rows):
    print(' '.join(f'{V_up[env.sidx[(r,c)]]:6.3f}' if (r,c) not in env.obstacles else '  WALL'
                   for c in range(env.n_cols)))
assert np.allclose(V_up[term_mask], 0.0), '终止态价值必须为 0'
print('✅ 解析策略评估完成：直接解线性系统得到 V^π')

## 3 · 策略评估（迭代解）并对拍解析解

不求逆，改用**迭代** Bellman 期望备份 $V \leftarrow R^\pi + \gamma P^\pi V$ 直到收敛。这是 bootstrapping 的最简体现，也是大状态空间下唯一可行的办法（求逆是 $O(S^3)$）。它该收敛到**和解析解一样**的 $V^\pi$。

In [ ]:
def policy_eval_iterative(pi_probs, P, R, gamma, term_mask, theta=1e-12, max_iter=10000):
    S = P.shape[0]
    P_pi, R_pi = policy_to_matrix(pi_probs, P, R)
    V = np.zeros(S)
    for k in range(max_iter):
        V_new = R_pi + gamma * (P_pi @ V)
        V_new[term_mask] = 0.0
        if np.max(np.abs(V_new - V)) < theta:
            V = V_new; break
        V = V_new
    return V, k + 1

V_up_iter, n_iter = policy_eval_iterative(pi_up, P, R, env.gamma, term_mask)
print(f'迭代策略评估收敛于 {n_iter} 次')
print('迭代解与解析解最大差:', np.max(np.abs(V_up_iter - V_up)))
assert np.allclose(V_up_iter, V_up, atol=1e-8), '迭代评估必须对拍解析评估'
print('✅ 对拍通过：迭代 Bellman 备份收敛到解析精确解（bootstrapping 有效）')

## 4 · 价值迭代：反复施加 Bellman 最优备份

把最优方程当赋值语句反复施加：$V(s) \leftarrow \max_a \sum_{s'} P(s'|s,a)[R + \gamma V(s')]$。

向量化：`Q = R + γ (P @ V)` 得到 `(S,A)` 的 Q 表，再 `V = Q.max(axis=1)`。记录每次的最大变化量 Δ，它应几何下降。

In [ ]:
def value_iteration(P, R, gamma, term_mask, theta=1e-12, max_iter=10000):
    S, A, _ = P.shape
    V = np.zeros(S)
    deltas = []
    for k in range(max_iter):
        Q = R + gamma * (P @ V)          # (S,A): 对每个 (s,a) 做一步备份
        V_new = Q.max(axis=1)            # 最优备份：取最好的动作
        V_new[term_mask] = 0.0
        delta = np.max(np.abs(V_new - V))
        deltas.append(delta)
        V = V_new
        if delta < theta:
            break
    return V, deltas

def greedy_policy(V, P, R, gamma):
    Q = R + gamma * (P @ V)
    return Q.argmax(axis=1), Q

V_star, deltas = value_iteration(P, R, env.gamma, term_mask)
pi_star, Q_star = greedy_policy(V_star, P, R, env.gamma)
print(f'价值迭代收敛于 {len(deltas)} 次')
print('最优价值 V*：')
for r in range(env.n_rows):
    print(' '.join(f'{V_star[env.sidx[(r,c)]]:6.3f}' if (r,c) not in env.obstacles else '  WALL'
                   for c in range(env.n_cols)))
arrows = {0: '^', 1: '>', 2: 'v', 3: '<'}
print('最优策略 π*：')
for r in range(env.n_rows):
    row = []
    for c in range(env.n_cols):
        s = (r, c)
        if s in env.obstacles: row.append('X')
        elif env.is_terminal(s): row.append('*')
        else: row.append(arrows[pi_star[env.sidx[s]]])
    print(' '.join(row))
# 关键检验：陷阱(1,3)下方的(2,3)最优动作应是「左」——绕开滑进陷阱的风险
assert arrows[pi_star[env.sidx[(2, 3)]]] == '<', '(2,3) 应向左绕开陷阱'
print('\n✅ VI 解出最优策略；注意 (2,3) 选「左」绕开 -1 陷阱 —— 数学自动权衡了风险')

## 5 · 验证几何收敛：误差 ≤ γ^k

压缩映射理论保证 $\|V_k - V^*\|_\infty \le \gamma^k \|V_0 - V^*\|_\infty$，即每次迭代误差至少缩小到 γ 倍。

我们记录每次迭代到 $V^*$ 的真实误差，验证它确实被 $\gamma^k$ 这条几何直线压住。

In [ ]:
# 重跑 VI，记录每步 V 到 V* 的真实误差
V = np.zeros(env.S)
errs = []
for k in range(60):
    errs.append(np.max(np.abs(V - V_star)))
    Q = R + env.gamma * (P @ V)
    V = Q.max(axis=1)
    V[term_mask] = 0.0
errs = np.array(errs)
gamma_bound = errs[0] * env.gamma ** np.arange(len(errs))
print(f"{'iter':>4} {'真实误差':>12} {'γ^k 上界':>12}")
for k in [0, 1, 2, 5, 10, 20]:
    print(f'{k:>4} {errs[k]:>12.2e} {gamma_bound[k]:>12.2e}')
# 真实误差必须始终 <= γ^k 上界（压缩映射保证）
assert np.all(errs <= gamma_bound + 1e-12), '误差应被 γ^k 几何上界压住'
# 收敛确实是几何的：相邻误差比 ≈ γ（取中段稳定区）
ratios = errs[6:12] / errs[5:11]
assert np.all(ratios <= env.gamma + 1e-6), '相邻误差比应 ≤ γ'
print('\n✅ 收敛是几何速率：误差被 γ^k 压住，相邻比 ≤ γ —— 压缩映射理论得证')

## 6 · 策略迭代：评估 + 改进交替

另一条路线：维护策略，反复「评估到收敛 → 贪心改进」，直到策略不变。

策略改进定理保证每轮不会变差；有限 MDP 上有限步收敛。它应得到**和 VI 完全一样**的 $V^*$ 和 $\pi^*$。

In [ ]:
def policy_iteration(P, R, gamma, term_mask, seed=0):
    S, A, _ = P.shape
    rng = np.random.default_rng(seed)
    pi = rng.integers(0, A, size=S)          # 随机初始确定性策略
    n_rounds = 0
    while True:
        # —— 策略评估：把当前确定性策略转 one-hot，解析求 V^π ——
        pi_probs = np.zeros((S, A)); pi_probs[np.arange(S), pi] = 1.0
        V = policy_eval_analytic(pi_probs, P, R, gamma, term_mask)
        # —— 策略改进：对 V^π 贪心 ——
        Q = R + gamma * (P @ V)
        pi_new = Q.argmax(axis=1)
        n_rounds += 1
        if np.array_equal(pi_new, pi):       # 策略稳定 -> 收敛
            return pi, V, n_rounds
        pi = pi_new

pi_pi, V_pi, rounds = policy_iteration(P, R, env.gamma, term_mask)
print(f'策略迭代收敛于 {rounds} 轮（远少于 VI 的 {len(deltas)} 次迭代）')
print('PI 得到的 V*：')
for r in range(env.n_rows):
    print(' '.join(f'{V_pi[env.sidx[(r,c)]]:6.3f}' if (r,c) not in env.obstacles else '  WALL'
                   for c in range(env.n_cols)))
print('✅ 策略迭代收敛，且只用了极少轮数（每轮把评估做透）')

## 7 · VI 与 PI 殊途同归：对拍

VI 和 PI 是广义策略迭代(GPI)的两个极端（评估做一步 vs 做到底），但都收敛到**同一个** $V^*$ 和**同一个**最优策略。这是本模块最重要的对拍。

In [ ]:
# 价值对拍
max_v_diff = np.max(np.abs(V_star - V_pi))
print(f'max |V_VI - V_PI| = {max_v_diff:.2e}')
assert max_v_diff < 1e-8, 'VI 与 PI 必须收敛到同一个 V*'
# 策略对拍（忽略终止态）
non_term = ~term_mask
same_policy = np.array_equal(pi_star[non_term], pi_pi[non_term])
print('VI 与 PI 的最优策略是否一致:', same_policy)
assert same_policy, 'VI 与 PI 必须给出同一个最优策略'
# 再对拍一个独立基准：V* 必须满足 Bellman 最优方程（不动点）
Q_check = R + env.gamma * (P @ V_star)
residual = np.max(np.abs(V_star[non_term] - Q_check.max(axis=1)[non_term]))
print(f'Bellman 最优方程残差 |V* - max_a Q| = {residual:.2e}')
assert residual < 1e-8, 'V* 必须是 Bellman 最优算子的不动点'
print('\n✅ 三重对拍通过：VI==PI 的 V*、VI==PI 的 π*、V* 是 Bellman 最优不动点')

---
## ✏️ 练习 1：单步 Bellman 最优备份

实现 `bellman_optimality_backup(V, P, R, gamma)`：给定当前价值估计 `V`，做**一步**最优备份，返回新的 `V_new`（不处理终止态钉零，纯备份）。

即 $V_{new}(s) = \max_a \sum_{s'} P(s'|s,a)[R(s,a) + \gamma V(s')]$（注意 R 已是期望奖励 R[s,a]）。

In [ ]:
def bellman_optimality_backup(V, P, R, gamma):
    # TODO: 用 Q = R + gamma * (P @ V) 得到 (S,A)，再对动作取 max 返回 (S,)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
V_test = np.zeros(env.S)
V1 = bellman_optimality_backup(V_test, P, R, env.gamma)
assert V1.shape == (env.S,)
# 从全 0 备份一步：只有「一步内能到 +1 目标」的状态价值变正
# (0,2) 向右 0.8 到目标(+1)，故其备份值应≈ 0.8*1 + ... > 0
assert V1[env.sidx[(0, 2)]] > 0.5, '(0,2) 一步可达目标，价值应明显为正'
# 反复施加这个备份应收敛到 V*（即它就是 VI 的核心）
Vk = np.zeros(env.S)
for _ in range(200):
    Vk = bellman_optimality_backup(Vk, P, R, env.gamma)
    Vk[term_mask] = 0.0
assert np.allclose(Vk, V_star, atol=1e-8), '反复备份应收敛到 V*'
print('✅ 练习 1 通过：单步最优备份正确，反复施加收敛到 V*')

## ✏️ 练习 2：从 Q* 提取最优策略与最优价值

给定最优动作价值 `Q_star`（形状 `(S,A)`），实现两个函数：
- `policy_from_Q(Q)`：返回贪心策略 `argmax_a Q[s,a]`（形状 `(S,)`）；
- `V_from_Q(Q)`：返回 `V*(s) = max_a Q[s,a]`（形状 `(S,)`）。

并验证 `V_from_Q(Q_star)` 与第 4 节算出的 `V_star` 一致。

In [ ]:
def policy_from_Q(Q):
    # TODO: 返回每个状态的 argmax 动作
    raise NotImplementedError

def V_from_Q(Q):
    # TODO: 返回每个状态的 max Q
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
pi_from_q = policy_from_Q(Q_star)
V_from_q = V_from_Q(Q_star)
assert np.array_equal(pi_from_q[~term_mask], pi_star[~term_mask]), '应与 VI 的贪心策略一致'
assert np.allclose(V_from_q[~term_mask], V_star[~term_mask], atol=1e-9), 'max_a Q* 应等于 V*'
print('✅ 练习 2 通过：V*=max_a Q*，π*=argmax_a Q* —— 有 Q 就不需要模型来做决策')

## ✏️ 练习 3：折扣因子 γ 如何改变最优策略

γ 编码远见。短视的 agent（小 γ）可能选择不同的路线。

实现 `solve_for_gamma(gamma)`：用给定 γ 重建 (P,R) 并跑价值迭代，返回最优策略数组。
然后比较 γ=0.9 与 γ=0.1 下起点附近策略是否可能不同（小 γ 更急于止损/避免每步 -0.04）。

In [ ]:
def solve_for_gamma(gamma):
    # TODO: 用 GridWorld(gamma=gamma) 建环境，build_PR 得到 P,R，
    #       构造 term_mask，跑 value_iteration，再 greedy_policy 返回策略数组
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
pi_far = solve_for_gamma(0.99)      # 极有远见
pi_near = solve_for_gamma(0.1)      # 极短视
assert pi_far.shape == (env.S,) and pi_near.shape == (env.S,)
# 远见策略里 (2,3) 仍应绕开陷阱（向左）
arrows = {0: '^', 1: '>', 2: 'v', 3: '<'}
assert arrows[pi_far[env.sidx[(2, 3)]]] == '<', '远见 agent 仍绕开陷阱'
# 两种 γ 至少在某个非终止状态上策略不同（远见 vs 短视行为有别）
diff = np.sum(pi_far[~term_mask] != pi_near[~term_mask])
print(f'γ=0.99 与 γ=0.1 下，{diff} 个状态的最优动作不同')
assert diff >= 1, 'γ 改变应导致至少一个状态的最优动作不同'
print('✅ 练习 3 通过：折扣因子 γ 确实塑造最优策略（远见程度影响决策）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def bellman_optimality_backup(V, P, R, gamma):
    Q = R + gamma * (P @ V)        # (S,A)
    return Q.max(axis=1)

In [ ]:
# 练习 2 参考答案
def policy_from_Q(Q):
    return Q.argmax(axis=1)

def V_from_Q(Q):
    return Q.max(axis=1)

In [ ]:
# 练习 3 参考答案
def solve_for_gamma(gamma):
    e = GridWorld(gamma=gamma)
    Pg, Rg = build_PR(e)
    tm = np.array([e.is_terminal(s) for s in e.states])
    Vg, _ = value_iteration(Pg, Rg, gamma, tm)
    pig, _ = greedy_policy(Vg, Pg, Rg, gamma)
    return pig

---
## 🧪 真实数据胶囊：用 MDP 解一个「学习资源分配」决策

MDP 不只解迷宫。考虑一个**真实风味**的序贯决策：每天你有精力做「复习(R)」或「学新(N)」，知识水平离散为 5 级 `0..4`。

- 学新：0.6 升一级、0.4 不变；即时奖励 0（投资未来）。
- 复习：巩固，0.9 留在原级、0.1 掉一级（遗忘）；即时奖励 = 当前级别（产出与水平成正比）。
- γ=0.95（有耐心）。问：什么时候该学新、什么时候该复习产出？用 VI 解出最优策略。

In [ ]:
def build_study_mdp(gamma=0.95):
    levels = 5
    S, A = levels, 2                 # 状态=知识级别 0..4；动作 0=学新 N, 1=复习 R
    P = np.zeros((S, A, S)); Rew = np.zeros((S, A))
    for s in range(levels):
        # 动作 0 学新：0.6 升级(封顶 4)，0.4 不变；奖励 0
        up = min(s + 1, levels - 1)
        P[s, 0, up] += 0.6; P[s, 0, s] += 0.4
        Rew[s, 0] = 0.0
        # 动作 1 复习：0.9 留原级，0.1 掉级(底 0)；奖励=当前级别
        down = max(s - 1, 0)
        P[s, 1, s] += 0.9; P[s, 1, down] += 0.1
        Rew[s, 1] = float(s)
    return P, Rew, gamma

Ps, Rs, g = build_study_mdp()
assert np.allclose(Ps.sum(axis=2), 1.0)
no_term = np.zeros(5, dtype=bool)        # 持续任务，无终止态
Vs, _ = value_iteration(Ps, Rs, g, no_term)
pis, _ = greedy_policy(Vs, Ps, Rs, g)
names = {0: '学新N', 1: '复习R'}
print('知识级别 -> 最优动作 (及其价值)：')
for s in range(5):
    print(f'  级别 {s}: {names[pis[s]]:6s}  V*={Vs[s]:.2f}')
# 直觉：低级别该学新(攒本钱)，高级别该复习(变现)
assert pis[0] == 0, '最低级别应学新（产出太低，先投资）'
assert pis[4] == 1, '最高级别应复习变现（已封顶，无可升）'
print('\n✅ MDP 自动得出「先投资再变现」的最优策略 —— 序贯决策的威力')

**🧪 胶囊练习**：实现 `crossover_level(gamma)`：在给定 γ 下解上面的 study MDP，返回**第一个**最优动作为「复习(1)」的最低级别（即「从攒本钱切换到变现」的临界级别）。直觉：γ 越大越有耐心、越晚变现（临界级别越高）。

In [ ]:
def crossover_level(gamma):
    # TODO: build_study_mdp(gamma) -> value_iteration -> greedy_policy
    #       返回最小的 s 使得最优动作==1；若没有返回 5
    raise NotImplementedError

In [ ]:
# 自测
lo = crossover_level(0.5)        # 短视：早点变现
hi = crossover_level(0.95)       # 有耐心：晚点变现
print(f'γ=0.5 临界级别={lo}，γ=0.95 临界级别={hi}')
assert 0 <= lo <= 5 and 0 <= hi <= 5
assert hi >= lo, 'γ 越大越有耐心，切换到变现的级别不应更低'
print('✅ 胶囊练习通过：折扣因子越大，越晚从投资切换到变现')

In [ ]:
# 📖 胶囊参考答案
def crossover_level(gamma):
    Pg, Rg, g = build_study_mdp(gamma)
    Vg, _ = value_iteration(Pg, Rg, g, np.zeros(5, dtype=bool))
    pig, _ = greedy_policy(Vg, Pg, Rg, g)
    for s in range(5):
        if pig[s] == 1:
            return s
    return 5

### 小结
- **MDP** = (S, A, P, R, γ)；马尔可夫性让价值不依赖历史，是一切的地基。
- **Bellman 期望方程**（线性，评估给定策略）vs **最优方程**（非线性含 max，求最优）。
- **价值迭代**：反复最优备份；Bellman 算子是 γ-压缩，**几何速率收敛**到唯一 V*。
- **策略迭代**：评估+改进交替，**有限步收敛**；与 VI 殊途同归到同一个 V* 与 π*。
- 全程**对拍**：迭代==解析、VI==PI、V*==Bellman 不动点。

下一站：**模块 02 · 时序差分与 Q-learning** —— 拿掉「已知模型」这个假设，只靠采样学到最优。